In [1]:
# ==========================================================
# Bronze Layer
# Import Libraries
# ==========================================================

from pyspark.sql import functions as F

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 3, Finished, Available, Finished, False)

In [2]:
# ==========================================================
# Paths
# ==========================================================

RAW_PATH = "Files/raw"

BRONZE_PATH = "Tables"

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 4, Finished, Available, Finished, False)

In [3]:
customers = (

    spark.read

    .option("header", True)

    .option("inferSchema", True)

    .csv(f"{RAW_PATH}/customers.csv")

)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 5, Finished, Available, Finished, False)

In [4]:
display(customers)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 168d135a-9dc6-4091-b1dd-d578482a3c9f)

In [5]:
customers.printSchema()

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 7, Finished, Available, Finished, False)

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BirthDate: date (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: long (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- LoyaltyLevel: string (nullable = true)
 |-- RegistrationChannel: string (nullable = true)
 |-- EmailVerified: boolean (nullable = true)
 |-- IsActive: boolean (nullable = true)
 |-- JoinDate: date (nullable = true)



In [6]:
(

    customers.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("bronze_customers")

)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 8, Finished, Available, Finished, False)

In [7]:
spark.sql("SHOW TABLES").show()

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 9, Finished, Available, Finished, False)

+--------------------+----------------+-----------+
|           namespace|       tableName|isTemporary|
+--------------------+----------------+-----------+
|`Retail Analytics...|bronze_customers|      false|
+--------------------+----------------+-----------+



In [8]:
# ==========================================================
# Generic Bronze Loader
# ==========================================================

def load_to_bronze(file_path, table_name):

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    (
        df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(table_name)
    )

    print(f"✓ {table_name} loaded successfully.")

    return df

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 10, Finished, Available, Finished, False)

In [9]:
customers = load_to_bronze(
    "Files/raw/customers.csv",
    "bronze_customers"
)

products = load_to_bronze(
    "Files/raw/products.csv",
    "bronze_products"
)

stores = load_to_bronze(
    "Files/raw/stores.csv",
    "bronze_stores"
)

employees = load_to_bronze(
    "Files/raw/employees.csv",
    "bronze_employees"
)

returns = load_to_bronze(
    "Files/raw/returns.csv",
    "bronze_returns"
)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 11, Finished, Available, Finished, False)

✓ bronze_customers loaded successfully.
✓ bronze_products loaded successfully.
✓ bronze_stores loaded successfully.
✓ bronze_employees loaded successfully.
✓ bronze_returns loaded successfully.


In [10]:
spark.sql("""
SHOW TABLES
""").show(truncate=False)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 12, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
+--------------------------------------+----------------+-----------+



In [11]:
# ==========================================================
# Load Sales Files
# ==========================================================

sales = (

    spark.read

    .option("header", True)

    .option("inferSchema", True)

    .csv("Files/raw/sales/*.csv")

)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 13, Finished, Available, Finished, False)

In [12]:
display(sales)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3b06400f-1102-4265-9ea3-cfbd9faa5503)

In [13]:
sales.printSchema()

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 15, Finished, Available, Finished, False)

root
 |-- SaleID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- EmployeeID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- SalesChannel: string (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitCost: integer (nullable = true)
 |-- CustomerCity: string (nullable = true)
 |-- StoreCity: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Cost: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Tax: double (nullable = true)



In [14]:
sales.count()

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 16, Finished, Available, Finished, False)

500000

In [15]:
(
    sales.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("bronze_sales")
)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 17, Finished, Available, Finished, False)

In [16]:
spark.sql("""
SHOW TABLES
""").show(truncate=False)

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 18, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_sales    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
+--------------------------------------+----------------+-----------+



In [17]:
tables = [

    "bronze_customers",

    "bronze_products",

    "bronze_stores",

    "bronze_employees",

    "bronze_returns",

    "bronze_sales"

]

for table in tables:

    count = spark.table(table).count()

    print(f"{table:<25} {count:,}")

StatementMeta(, def656d3-4d79-4b73-958d-6500017af7b5, 19, Finished, Available, Finished, False)

bronze_customers          1,000
bronze_products           300
bronze_stores             20
bronze_employees          258
bronze_returns            71,486
bronze_sales              500,000
